# Silver Layer Transformation 

In the Silver layer, the Bronze datasets are cleaned and standardized to ensure data consistency and usability for downstream analytics. String fields are trimmed, special characters like | are removed, and placeholder values such as N/A, NA, or empty strings are replaced with NULL. Country names are normalized; for example, UK, U.K., and misspelled variants like United Kindom are standardized to United Kingdom. Monetary fields like unit_price, cost, and sales are converted from strings with currency symbols into numeric values, while a separate currency column stores the currency type. Duplicate records are eliminated using the appropriate primary keys, and an ingestion_timestamp column is added to record when the data was processed in the Silver layer.

# Silver Table (Region)

In [0]:
%sql
SELECT * FROM project.bronze.region;

In [0]:
# Load bronze.region table
df_region = spark.table("project.bronze.region")
display(df_region)
df_region.printSchema()

In [0]:
# Check that 'sales_territory_key' is unique
# Output should be empty
display(spark.sql("""
                  SELECT 
                    sales_territory_key, 
                    COUNT(*) AS cnt
                  FROM project.bronze.region
                  GROUP BY sales_territory_key
                  HAVING cnt > 1
                  """))

In [0]:
from pyspark.sql import functions as F

# Trim string columns
string_cols = [c for c, t in df_region.dtypes if t == "string"]
for c in string_cols:
    df_region.withColumn(c, 
                            F.when(F.upper(c).isin("N/A", "NA", "NULL", "NONE", "-", ""), None)
                            .otherwise(F.col(c)))  \
             .withColumn(c, F.trim(F.col(c)))

# Replace 'USA' with 'United States'
df_region = df_region.withColumn(
    "country",
    F.when(F.upper(F.col("country")) == F.lit("USA"), "United States")
     .otherwise(F.col("country"))
)

# Drop ingestion timestamp column
df_region = df_region.drop("source_file", "ingestion_timestamp")

# Sanity check
display(df_region)

In [0]:
# Write silver region table
(df_region.write
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .format("delta")
            .saveAsTable("project.silver.region")
)

# Sanity check
display(spark.table("project.silver.region"))

# Silver Table (Salesperson_region)

In [0]:
%sql 
SELECT * FROM project.bronze.salesperson_region;

In [0]:
df_salesperson_region = spark.table("project.bronze.salesperson_region")
display(df_salesperson_region)
df_salesperson_region.printSchema()

In [0]:
# Check if there are any null values in the keys
display(spark.sql("""
                  SELECT *
                  FROM project.bronze.salesperson_region
                  WHERE employee_key IS NULL OR sales_territory_key IS NULL
                  """))

In [0]:
# Check if 'employee_key' is unique
display(spark.sql("""
                  SELECT employee_key, COUNT(sales_territory_key) AS territory_cnt
                  FROM project.bronze.salesperson_region
                  GROUP BY employee_key
                  """))

In [0]:
# Check if {employee_key, sales_territory_key} is unique
# If no output -> {employee_key, sales_territory_key} = Primary Key

display(spark.sql("""
                  SELECT COUNT(*) AS employee_territory_cnt
                  FROM project.bronze.salesperson_region
                  GROUP BY employee_key, sales_territory_key
                  HAVING COUNT(*) > 1
                  """))

In [0]:
# Drop unwanted columns
df_salesperson_region = df_salesperson_region.drop("source_file", "ingestion_timestamp")

# Write silver salesperson_region table
(df_salesperson_region.write
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .format("delta")
            .saveAsTable("project.silver.salesperson_region")
)

# Sanity check
display(spark.table("project.silver.salesperson_region"))

# Silver Table (Salesperson)

In [0]:
%sql
SELECT * FROM project.bronze.salesperson;

In [0]:
df_salesperson = spark.table("project.bronze.salesperson")
display(df_salesperson)
df_salesperson.printSchema()

In [0]:
# Drop unwanted columns
df_salesperson = df_salesperson.drop("source_file", "ingestion_timestamp")

# Strip and capitalize string columns
string_cols = [c for c, t in df_salesperson.dtypes if t == "string"]
for c in string_cols:
    df_salesperson = df_salesperson.withColumn(c, 
                                         F.when(F.upper(c).isin("N/A", "NA", "NULL", "NONE", "-", ""), None)
                                            .otherwise(F.col(c)))  \
                                   .withColumn(c, F.trim(F.col(c))) \
                                   .withColumn(c, F.initcap(F.col(c)))

In [0]:
# Check total count and compare to distinct 'employee_key' and distinct 'employee_id' counts
display(spark.sql("""
                  SELECT
                    COUNT(*) AS total_cnt
                  FROM project.bronze.salesperson
                  """))

display(spark.sql("""
                  SELECT
                    COUNT(DISTINCT employee_key) AS distinct_empl_key_cnt
                  FROM project.bronze.salesperson
                  """))

display(spark.sql("""
                  SELECT
                    COUNT(DISTINCT employee_id) AS distinct_empl_id_cnt
                  FROM project.bronze.salesperson
                  """))

In [0]:
# Check that 'employee_key' and 'employee_id' are not null
# Count should be zero
display(spark.sql("""
                  SELECT
                    COUNT(*) AS key_id_null_cnt
                  FROM project.bronze.salesperson
                  WHERE employee_key IS NULL OR employee_id IS NULL
                  """))

In [0]:
# Write silver salesperson table
(df_salesperson.write
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .format("delta")
            .saveAsTable("project.silver.salesperson")
)

# Sanity check
display(spark.table("project.silver.salesperson"))

In [0]:
r"""
from pyspark.sql import functions as F

# Load Bronze table
bronze_salesperson = spark.table("workspace.default.bronze_salesperson")

# Trim string columns (safe)
df = (bronze_salesperson.withColumn("employeekey", F.trim(F.col("employeekey"))).withColumn("salesperson", F.trim(F.col("salesperson"))).withColumn("title", F.trim(F.col("title"))).withColumn("upn", F.trim(F.col("upn"))))

df = df.withColumn( "employeeid",F.expr("try_cast(nullif(trim(cast(employeeid as string)), '') as bigint)"))

# Normalize UPN (email) to lowercase plus null if empty after trim
df = df.withColumn("upn",F.when(F.col("upn").isNull() | (F.col("upn") == ""), None).otherwise(F.lower(F.col("upn"))))

# Remove rows with null key
df = df.filter(F.col("employeeid").isNotNull())

# Remove duplicates (natural key = employeeid)
df = df.dropDuplicates(["employeeid"])

# Reorder columns 
df = df.select(
    "employeekey",
    "employeeid",
    "salesperson",
    "title",
    "upn",
    "ingestion_timestamp"
)

# Save Silver table
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.default.silver_salesperson")

# Preview 
display(df)
"""

# Silver Table (Reseller)

In [0]:
%sql
SELECT * FROM project.bronze.reseller;

In [0]:
df_reseller = spark.table("project.bronze.reseller")
display(df_reseller)
df_reseller.printSchema()

In [0]:
# Drop unwanted columns
df_reseller = df_reseller.drop("source_file", "ingestion_timestamp")

# Strip and capitalize string columns
string_cols = [c for c, t in df_reseller.dtypes if t == "string"]
for c in string_cols:
    df_reseller = df_reseller.withColumn(c, 
                                         F.when(F.upper(c).isin("N/A", "NA", "NULL", "NONE", "-", ""), None)
                                            .otherwise(F.col(c))) \
                             .withColumn(c, F.trim(F.col(c))) \
                             .withColumn(c, F.initcap(F.col(c)))

In [0]:
# Check distinct 'state_province' and 'country_region'
display(spark.sql("""
                  SELECT DISTINCT state_province
                  FROM project.bronze.reseller
                  """))

display(spark.sql("""
                  SELECT DISTINCT country_region
                  FROM project.bronze.reseller
                  """))

In [0]:
# Replace country abbreviations with country name
# We have already capitalize 'country_region'
df_reseller = df_reseller.replace(
    {"Usa": "United States",
     "Us": "United States",
     "U.s.a.": "United States",
     "U.k.": "United Kingdom",
     "Uk": "United Kingdom"},
    subset=["country_region"]
)

# Sanity check
display(df_reseller.select("country_region").distinct())

In [0]:
# Check if 'reseller_key' is null
# Count should be zero
display(spark.sql("""
                  SELECT COUNT(*) AS null_reseller_key_cnt
                  FROM project.bronze.reseller
                  WHERE reseller_key IS NULL
                  """))

# Chech if 'reseller_key' is unique
# Result should be empty
display(spark.sql("""
                  SELECT
                    reseller_key,
                    COUNT(*) AS cnt
                  FROM project.bronze.reseller
                  GROUP BY reseller_key
                  HAVING COUNT(*) > 1
                  """))

In [0]:
display(spark.sql("""
                  SELECT count(*) AS total_cnt
                  FROM project.bronze.reseller
                  """))
display(spark.sql("""
                  SELECT count(DISTINCT reseller_key) AS distinct_reseller_key_cnt
                  FROM project.bronze.reseller
                  """))

In [0]:
# Write silver reseller table
(df_reseller.write
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .format("delta")
            .saveAsTable("project.silver.reseller")
)

# Sanity check
display(spark.table("project.silver.reseller"))

# Silver Table (Targets)

In [0]:
%sql
SELECT * FROM project.bronze.targets;

In [0]:
df_targets = spark.table("project.bronze.targets")
display(df_targets)
df_targets.printSchema()

In [0]:
from pyspark.sql import functions as F

# Drop unwanted columns
df_targets = df_targets.drop("source_file", "ingestion_timestamp")

# Trim string columns
df_targets = (df_targets.withColumn("target", F.trim("target"))
                        .withColumn("target_month", F.trim("target_month"))
                        )


In [0]:
# Check currency codes available in 'target' column
# Check both start and end of string
display(spark.sql("""
                  SELECT
                    DISTINCT RIGHT(target, 1) AS currency_code_right
                  FROM project.bronze.targets
                  """))

display(spark.sql("""
                  SELECT
                    DISTINCT LEFT(target, 1) AS currency_code_left
                  FROM project.bronze.targets
                  """))

In [0]:
# Check for nulls in 'target' column
display(spark.sql("""
                  SELECT *
                  FROM project.bronze.targets
                  WHERE target IS NULL
                  """))

In [0]:
# Map currency sign to currency code
df_targets = df_targets.withColumn("currency",
                                F.when(F.col("target").rlike("\\$"), F.lit("USD"))
                                .when(F.col("target").rlike("€"), F.lit("EUR"))
                                .when(F.col("target").rlike("£"), F.lit("GBP"))
                                .otherwise(F.lit("USD"))
                            )

# Clean 'target' column and cast to decimal
df_targets = df_targets.withColumn("target",
    F.regexp_replace(F.col("target"), r"[$,€£\s]", "").cast("decimal(18,2)"))

In [0]:
# Check if 'target_month' is null
# Count should be zero
display(spark.sql("""
                  SELECT COUNT(*) AS null_target_month_cnt
                  FROM project.bronze.targets
                  WHERE target_month IS NULL
                  """))

# Inspect 'target_month' column
display(spark.sql("""
                  SELECT DISTINCT target_month
                  FROM project.bronze.targets
                  """))

In [0]:
# Convert string month to date dtype month
df_targets = df_targets.withColumn("targetmonth",
                            F.regexp_replace("target_month", "^[A-Za-z]+,\\s*", "")) \
                        .withColumn("target_month_clean",
                            F.coalesce(
                                F.to_date("targetmonth", "MMMM d, yyyy"),
                                F.to_date("targetmonth", "MMM d, yyyy")
                            ))

# Sanity check
display(df_targets)

In [0]:
# Check if employee ID is null
display(spark.sql("""
                  SELECT COUNT(*) AS null_employee_id_cnt 
                  FROM project.bronze.targets
                  WHERE employee_id IS NULL"""))

In [0]:
# Count of targets per 'employee_id'
display(spark.sql("""
                  SELECT employee_id, COUNT(target) AS target_cnt
                  FROM project.bronze.targets
                  GROUP BY employee_id
                  ORDER BY target_cnt DESC
                  """))

In [0]:
# Check for duplicate records
df_targets.groupBy(df_targets.columns).count().filter(F.col("count") > 1).show()

In [0]:
# Keep columns to save
df_targets = df_targets.drop("target_month", "targetmonth") \
                       .withColumnRenamed("target_month_clean", "target_month") \
                       .select("employee_id", "target", "currency", "target_month")

# Write silver table
(
    df_targets.write
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .format("delta")
          .saveAsTable("project.silver.targets")
)

# Sanity check
display(spark.table("project.silver.targets"))

# ↓ **INSERT HERE** ↓

# Final check

In [0]:
# Show tables ingested in silver
display(spark.sql("SHOW TABLES IN project.silver"))